# News article category classifier

My collector pulls RSS feeds from 97 news sources into MongoDB. Every
article arrives with a title, usually a summary, and whatever section the
publisher filed it under. This notebook trains a model that reads the headline
and predicts one of a fixed set of topics.

The plan:

1. **Analyse the data** - what have I actually got?
2. **Clean it** - strip the furniture, drop what isn't a news article.
3. **Label and split** - where 4,301 hand labels came from, then train / val / test.
4. **Run the models** - four of them, dumbest first.
5. **Learning curve** - is more labelling still buying me anything?
6. **Metrics** - precision, recall, F1, worked out by hand from the counts.
7. **Graphs** - confusion matrix, ROC, and where to put the cut.
8. **Conclusion** - leakage checks, and what the number isn't.

All the real code lives in `src/newsml/` with unit tests behind it. The notebook
only calls it, so there is no number here I can't reproduce from a script.

It also reads a **frozen snapshot**, not the live database. The corpus grows by
the hour, so a number computed against it is true for an afternoon and then
quietly stops being reproducible. The snapshot id is printed below and every
figure here is a function of it. Run the cells top to bottom.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from newsml.config import SEED, SNAPSHOT_DIR, TAXONOMY_PATH
from newsml.dataset import from_snapshot
from newsml.labels import load_taxonomy
from newsml.load import load_articles
from newsml.models import LADDER, evaluate
from newsml.snapshot import read as read_snapshot

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": False})

# Everything below reads this frozen snapshot rather than the live collector,
# which grows by the hour. Re-run this notebook next month against a database
# twice the size and every number in it stays where it is.
SNAPSHOT_ID = "20260825-121"
snap = read_snapshot(SNAPSHOT_DIR / SNAPSHOT_ID)

for field, value in snap.provenance.items():
    print(f"{field:<18}{value}")

## 1. Analysing the data

First question, before any modelling: how many articles, from where, how much
text does each one carry, and how far back do they go. Reading Mongo here is
read-only. The notebook never writes to the collector's database.

In [ ]:
from datetime import datetime

taxonomy = load_taxonomy(TAXONOMY_PATH)

# The snapshot's own cut, reapplied. Sections 1 and 2 need the raw article
# fields, which the snapshot does not keep, so they read Mongo directly - but
# through the same window, so they describe exactly the corpus that was frozen
# rather than whatever has arrived since.
cut = datetime.fromisoformat(snap.manifest["collected_before"])
articles = load_articles(collected_before=cut)

print(f"{len(articles):,} articles collected before {cut:%Y-%m-%d %H:%M}")
print(f"{len({a.source_name for a in articles})} sources")
print(f"published {min(a.published_at for a in articles):%Y-%m-%d}"
      f" to {max(a.published_at for a in articles):%Y-%m-%d}")
print(f"taxonomy v{taxonomy.version}: {len(taxonomy.classes)} classes")

In [ ]:
# A field that is often empty can't be used as a feature, so count them first.
for name in ("title", "summary", "content", "categories"):
    filled = sum(1 for a in articles if getattr(a, name))
    print(f"{name:<11}{filled:>7,}{filled / len(articles):>8.1%}")

sources = Counter(a.source_name for a in articles)
print("\nbusiest sources:")
for name, n in sources.most_common(8):
    print(f"  {name:<38}{n:>6,}")

In [ ]:
words = [len(a.text("title_summary").split()) for a in articles]
per_day = Counter(a.published_at.date() for a in articles)
recent = sorted(per_day)[-14:]

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.2))

left.bar([f"{d:%d %b}" for d in recent], [per_day[d] for d in recent], color="#3b3b3b")
left.set_title("Articles published per day (last 14 days)", loc="left")
left.tick_params(axis="x", rotation=60)

right.hist(words, bins=40, color="#3b3b3b")
right.axvline(float(np.median(words)), color="#c02a2a", lw=1.2,
              label=f"median {np.median(words):.0f} words")
right.set_title("Length of title + summary", loc="left")
right.set_xlabel("words")
right.legend(frameon=False)

for ax in (left, right):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### What that tells me

- The printed date range has a long tail back to 2008, but that is a handful of
  evergreen pages. The bar chart shows where the mass actually is: the last week
  or two. Wide enough to cut a split by time, nowhere near wide enough to say
  anything about how the model ages.
- Most articles are 20 to 60 words. This is **short-text classification**: the
  model gets a headline and a sentence, not an essay.
- `content` (the scraped body) exists for most articles but not all, and whether
  it exists depends on the *publisher*, not the topic. Train on it and the model
  learns which newspaper wrote the story. So everything below reads
  **title + summary** only.

## 2. Cleaning the data

Raw feed text is full of things that aren't the story: datelines
("NEW DELHI:"), wire credits ("(PTI)"), "Also Read" links, share prompts,
subscribe trailers. None of that is about the topic and all of it is specific to
one publisher, so leaving it in lets the model identify the newspaper instead of
the subject.

Two passes:

- `clean()` rewrites the text. Normalise unicode, fold curly quotes and dashes,
  strip the furniture, and pull the dateline city and the wire agency out into
  their own fields rather than deleting them.
- `partition()` decides what even counts as an article. Horoscopes, live blogs,
  scorecards, photo galleries, "top 10" listicles, weather bulletins, paywall
  stubs and anything under 12 words get dropped, each with a reason code.

In [ ]:
from newsml.admit import partition
from newsml.clean import clean

pairs = [(a, clean(a.text("title_summary"))) for a in articles]

# Pick one article that actually had something pulled out of it.
example, example_clean = next(
    ((a, c) for a, c in pairs if c.wire_agency or c.dateline_city),
    pairs[0],
)

print("BEFORE\n", example.text("title_summary")[:280], "\n")
print("AFTER\n", example_clean.text[:280], "\n")
print("pulled out ->  city:", example_clean.dateline_city or "-",
      "  wire:", example_clean.wire_agency or "-")

In [ ]:
# Language detection is off here: every source in the config is already an English feed.
admitted, rejected = partition(pairs, check_language=False)

print(f"kept     {len(admitted):>6,}")
print(f"rejected {len(rejected):>6,}   ({len(rejected) / len(pairs):.2%})\n")

reasons = dict(Counter(str(r.reason) for r in rejected).most_common())
for reason, n in reasons.items():
    print(f"  {reason:<22}{n:>5}")

fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.barh(list(reasons)[::-1], list(reasons.values())[::-1], color="#3b3b3b")
ax.set_title("Why articles were rejected", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Most of what gets dropped is stubs and articles carrying a broken timestamp, not
the content rules. The format rules themselves only account for a couple of
percent of the corpus, which is the intent: this filter is a scalpel, not a net,
and every pattern in it was written after reading the articles it matched.

The one that surprised me was weather and traffic bulletins. They are real prose
from a real newsroom, so no length or format rule catches them, but they aren't
*about* anything, and when I hand-labelled a sample I filed every single one as
unsorted. That is the tell: if a human can't give it a topic, it shouldn't be in
the training set teaching the model one.

## 3. Where the labels came from

This is the part that took the longest, and the part the score depends on.

**The first attempt used weak labels** - a topic guessed from three signals the
feed already carries: the section an article arrived in (`Indian Express -
Technology` means technology), a prior for single-subject publishers (`The
Lancet` means health), and the publisher's own category tags.

Weak labels are free, and they are not good enough, for two separate reasons:

- **They only reach two thirds of the corpus.** A general national desk names no
  topic, and a tag like `india` or `chennai` names a *place*, not a subject.
- **No publisher runs a "Crime" or a "War" RSS section.** Those topics are
  structurally invisible to this method no matter how many articles I collect.

So the labels below are **human**. Three rounds, 4,301 articles.

Round 3 is worth reading as a result in its own right. It was aimed squarely at
the three weakest classes, came back clean, and **moved nothing** - macro-F1
went 0.678 to 0.671. That is not a wasted round: it is the evidence that more
labels have stopped being the lever, which is a thing you can only learn by
buying one more round and watching it fail.

In [ ]:
# Every human label, frozen inside the snapshot beside the corpus it belongs to.
# Three rounds; no round ever re-asked an article an earlier one had settled.
gold = dict(snap.labels)
print(f"{len(gold):,} hand-labelled articles across {len(set(gold.values()))} classes")

by_class = Counter(gold.values())
print(f"\nlargest {max(by_class.values())}   smallest {min(by_class.values())}")
for topic, n in by_class.most_common():
    print(f"  {topic:<24}{n:>5}")

### How those articles were chosen

Not at random, and that matters. Round 1 *was* random and it returned 20
`conflict_war` labels out of 1,200 - because that is roughly how often armed
conflict appears in these feeds. Scaling a random sample cannot fix that: at
1.7% prevalence, 150 labels of one class needs 9,000 draws from a corpus holding
about 7,500 distinct stories.

So round 2 went looking. Each class stated how many labels it still wanted, and
the articles most like the ones it already had were retrieved to fill the gap -
nearest-centroid over TF-IDF, weighted by which kind of source that class
actually turns up on. The sheet stayed blind: title and summary only, no source,
no proposed label, because an annotator shown a guess agrees with it.

The cell below is the check that the weak labels were worth replacing.

In [ ]:
from newsml.dataset import group_of_topic, weak_label

by_id = {a.article.id: a.article for a in admitted}
agree = Counter()
for article_id, human in gold.items():
    if (article := by_id.get(article_id)) is None:
        continue
    weak, source = weak_label(article, taxonomy)
    if source is None or weak == taxonomy.unsorted:
        agree["no weak label at all"] += 1
    elif weak == group_of_topic(human, taxonomy):
        agree["weak label agreed"] += 1
    else:
        agree["weak label disagreed"] += 1

total = sum(agree.values())
for label, n in agree.most_common():
    print(f"{label:<24}{n:>6,}{n / total:>8.1%}")

covered = total - agree["no weak label at all"]
print(f"\nwhere a weak label existed it agreed {agree['weak label agreed'] / covered:.1%}"
      f" of the time, at GROUP level")
print("that is the ceiling weak labels could ever have reached, and it is why")
print("everything below trains on the human labels instead.")

### Building the dataset

`from_snapshot()` joins the labels onto the rows and stops. It is what it does
*not* do that matters: the cleaning, the admission, the near-duplicate grouping
and the split boundary were all decided when the snapshot was written, so none
of them can drift between runs or between sections of this notebook.

Only human labels reach the model. The check above measured how often the weak
teacher agrees with a person, and that is not good enough to train on when the
model is then scored against people. There is no finer level left for a weak
label to compete with either: the taxonomy is fixed and flat (see the note at
the top of `taxonomy.yaml`), so each of these 13 classes is already the unit
the model predicts.

The split is **grouped and temporal** at the same time:

- grouped, so near-identical copies of the same wire story never land on both
  sides. Six papers run the same PTI copy. One in train and one in test is
  memorisation being scored as understanding. Where the line sits was
  **measured, not assumed**: 43 borderline pairs were hand-judged, and the
  threshold the project had been using recovered 7 of the 31 real duplicates.
  It now recovers 28. Reassuringly the score barely moved, which says the
  leakage it was letting through was real but small.
- temporal, so every test article was published after every training article.
  News vocabulary moves fast, and testing on the past means the model has
  already seen the future. The two cut times are chosen from the *labelled*
  articles and recorded in the snapshot manifest. Taking them over the whole
  corpus instead looked reasonable and put 37 labelled articles in a test split
  of 1,317, because labelling stops the day a round is drawn and everything
  collected afterwards is unlabelled.

A story cluster that straddles the cut can't satisfy both, so it gets dropped
whole and counted rather than quietly truncated.


In [ ]:
data = from_snapshot(snap, taxonomy, min_per_class=40)

for name, n in data.counts.items():
    print(f"{name:<22}{n:>7,}")

print(f"\n{len(data.classes)} classes: {', '.join(data.classes)}")
print("\n`unlabelled` is every article no human has reached yet - the pool the")
print("next labelling round draws from, not a failure.")

In [ ]:
dist = data.distribution("train")

fig, ax = plt.subplots(figsize=(6.6, 5.2))
ax.barh(list(dist)[::-1], list(dist.values())[::-1], color="#3b3b3b")
ax.set_xlabel("training articles")
ax.set_title("Class balance in train", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

biggest, smallest = max(dist.values()), min(dist.values())
print(f"imbalance {biggest / smallest:.1f}:1  (largest {biggest}, smallest {smallest})")
print(f"train {len(data.train):,}  val {len(data.val):,}  test {len(data.test):,}")

### Two checks before I train anything

These are `assert`s rather than prints on purpose. If the split is broken I want
the notebook to stop, not to carry on and print me a great score.

In [ ]:
train_groups = {e.group_id for e in data.train}
val_groups = {e.group_id for e in data.val}
test_groups = {e.group_id for e in data.test}

# No story cluster is allowed to appear in two splits.
assert not train_groups & val_groups, "a story is in both train and val"
assert not train_groups & test_groups, "a story is in both train and test"
assert not val_groups & test_groups, "a story is in both val and test"

# And nothing in test may predate anything in train.
newest_train = max(e.published_at for e in data.train)
oldest_test = min(e.published_at for e in data.test)
assert oldest_test >= newest_train, "a test article was published before a training one"

# Every label reaching the model came from a person.
assert {e.label_source for e in data.train} == {"human"}

print("no story spans two splits")
print(f"train ends   {newest_train:%Y-%m-%d %H:%M}")
print(f"test starts  {oldest_test:%Y-%m-%d %H:%M}")
print(f"{data.dropped_at_boundary} articles dropped for straddling a cut")

## 4. Running the models

Four models, in order of how much they are allowed to assume. Nothing gets
credit until something dumber has been tried first.

| # | model | what it does |
|---|---|---|
| 1 | `majority` | always answers with the most common class. This is the floor. |
| 2 | `complement_nb` | counts which words go with which class. Naive Bayes, the variant built for imbalanced text. |
| 3 | `tfidf_linear_svc` | draws the widest boundary it can between classes. |
| 4 | `hashing_sgd` | same idea with no stored vocabulary, so the shipped model stays small, and it gives a confidence score. |

**Macro-F1** is the number I care about: score each class on its own, then
average with every class counting once. Plain accuracy lets a model ignore all
the small classes and still look respectable, and this corpus stays imbalanced
enough that some of the 13 classes are thin even after rolling everything up.

Everything is scored on **validation**. The test split stays sealed until the
end of the project.

In [ ]:
results, fitted = {}, {}

for name in LADDER:
    result, model = evaluate(name, data, split="val")
    results[name], fitted[name] = result, model
    print(f"{name:<20} macro-F1 {result.macro_f1:.3f}   accuracy {result.accuracy:.3f}"
          f"   {result.fit_seconds:>5.2f}s to fit"
          f"   {result.predict_ms_per_doc:.3f} ms/article")

best = max(results, key=lambda name: results[name].macro_f1)
floor = results["majority"].macro_f1
print(f"\nbest rung: {best} at {results[best].macro_f1:.3f},"
      f" {results[best].macro_f1 - floor:.3f} above the floor")

In [ ]:
rungs = list(results)
spots = np.arange(len(rungs))

fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.bar(spots - 0.19, [results[r].macro_f1 for r in rungs], 0.38,
       label="macro-F1", color="#3b3b3b")
ax.bar(spots + 0.19, [results[r].accuracy for r in rungs], 0.38,
       label="accuracy", color="#b8b8b8")
ax.set_xticks(spots)
ax.set_xticklabels(rungs, rotation=12)
ax.set_ylim(0, 1)
ax.set_title("Every rung, scored on validation", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The `majority` rung is the reason macro-F1 is the headline. With 13 classes,
always guessing the biggest one still gets a healthy share right on accuracy
(see the number printed above), and scores almost nothing on macro-F1, because
its recall is exactly zero on 12 of the 13 classes by construction.


## 5. Learning curve - is more labelling still worth it?

One score from one fit doesn't tell me whether the model is finished learning.
So I retrain each rung ten times, every run on a bigger slice of the same
training set, and score every run on the same validation split.

Read the shape, not the height:

- **still climbing at run 10** - more hand labels will keep paying, go and label
  another round.
- **flat since run 5** - the model already has all the data it can use, and the
  next gain has to come from better features, a better model, or a change to the
  taxonomy.

The slices are stratified, so run 3 gets 30% of every class rather than 30% of
the corpus and none of the small classes. This is the single most useful chart
in the notebook, because labelling is the expensive part.

In [ ]:
from sklearn.metrics import f1_score

from newsml.models import build as build_model

x_train, y_train = data.xy("train")
x_val, y_val = data.xy("val")
labels = list(data.classes)

# Shuffle each class's row numbers once, so every run is a superset of the one before.
rng = np.random.default_rng(SEED)
by_class_rows: dict[str, list[int]] = {}
for i, topic in enumerate(y_train):
    by_class_rows.setdefault(topic, []).append(i)
for rows in by_class_rows.values():
    rng.shuffle(rows)

RUNS = 10
curve = {name: [] for name in LADDER}
sizes = []

for run in range(1, RUNS + 1):
    share = run / RUNS
    picked = [i for rows in by_class_rows.values()
              for i in rows[: max(3, round(len(rows) * share))]]
    sizes.append(len(picked))
    texts = [x_train[i] for i in picked]
    topics = [y_train[i] for i in picked]

    for name in LADDER:
        model = build_model(name)
        model.fit(texts, topics)
        score = f1_score(y_val, model.predict(x_val), labels=labels,
                         average="macro", zero_division=0)
        curve[name].append(float(score))

    line = "   ".join(f"{name} {curve[name][-1]:.3f}" for name in LADDER)
    print(f"run {run:>2}   {len(picked):>5} articles   {line}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.6))
for name in LADDER:
    ax.plot(range(1, RUNS + 1), curve[name], marker="o", ms=3.5, lw=1.4, label=name)
ax.set_xticks(range(1, RUNS + 1))
ax.set_xlabel("run  (each one trains on more articles than the last)")
ax.set_ylabel("macro-F1 on validation")
ax.set_title("Does more hand labelling still help?", loc="left")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)

# Second axis along the top just to say how many articles each run actually saw.
top = ax.twiny()
top.set_xlim(ax.get_xlim())
top.set_xticks(range(1, RUNS + 1))
top.set_xticklabels([f"{n:,}" for n in sizes], fontsize=7, rotation=45)
plt.tight_layout()
plt.show()

half, full = curve[best][RUNS // 2 - 1], curve[best][-1]
print(f"{best}: {half:.3f} at run {RUNS // 2}  ->  {full:.3f} at run {RUNS}")
print(f"so the second half of the training data was worth {full - half:+.3f} macro-F1")

That last number is the answer to "should I go and label more articles?".
Anything under about +0.01 means the curve has flattened and the next win has to
come from somewhere else - most likely from merging classes the model cannot
separate, rather than from more rows.

## 6. Metrics

Every metric anyone will ever quote about this model is some ratio of the counts
in one table, so it is worth reading them off by hand once instead of trusting
`classification_report` to have picked the right one.

There are 13 classes and no single "positive" one, so each class is scored
**against all the rest**: technology versus not-technology, then sport versus
not-sport, and so on. Four counts per class:

- **TP** - the diagonal cell. Filed here, belonged here.
- **FP** - the rest of that class's *column*. Other topics wrongly filed here.
- **FN** - the rest of that class's *row*. This topic filed somewhere else.
- **TN** - everything that never involved the class at all.

Then four ratios, three of which share the same TP on top and only disagree
about the denominator:

| metric | the question it answers | denominator |
|---|---|---|
| precision | of the articles I filed here, how many belonged? | the column, TP+FP |
| recall | of the articles that belonged here, how many did I find? | the row, TP+FN |
| specificity | of the articles that weren't this topic, how many did I leave alone? | TN+FP |
| F1 | harmonic mean of precision and recall | - |

F1 uses the harmonic mean rather than a plain average because the harmonic mean
is dragged down by whichever of the two is worse. A class that files one article
and gets it right has precision 1.0 and recall near zero. The plain average
calls that 0.5, F1 correctly calls it about 0.02.

In [ ]:
from sklearn.metrics import confusion_matrix

predicted = fitted[best].predict(x_val)
matrix = confusion_matrix(y_val, predicted, labels=labels)

tp = np.diag(matrix)
fp = matrix.sum(axis=0) - tp  # down the column: other topics called this one
fn = matrix.sum(axis=1) - tp  # across the row: this topic called something else
tn = matrix.sum() - tp - fp - fn


# Guard against a class with no predictions at all, which would divide by zero.
def ratio(top, bottom):
    return np.divide(top, bottom, out=np.zeros(len(labels)), where=bottom > 0)


precision = ratio(tp, tp + fp)
recall = ratio(tp, tp + fn)
specificity = ratio(tn, tn + fp)
f1 = ratio(2 * precision * recall, precision + recall)

head = (f"{'class':<24}{'n':>5}{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>6}"
        f"{'prec':>8}{'rec':>8}{'spec':>8}{'F1':>8}")
print(head)
print("-" * len(head))
for i in np.argsort(-f1):
    topic = labels[i]
    print(f"{topic:<24}{tp[i] + fn[i]:>5}{tp[i]:>5}{fp[i]:>5}{fn[i]:>5}{tn[i]:>6}"
          f"{precision[i]:>8.3f}{recall[i]:>8.3f}{specificity[i]:>8.3f}{f1[i]:>8.3f}")

# sklearn as the check on my arithmetic, not as the source of it.
assert np.allclose(f1, [results[best].per_class_f1[c] for c in labels])

thin = [labels[i] for i in range(len(labels)) if tp[i] + fn[i] < 10]
print(f"\nunder 10 validation articles, so these scores are noise: {', '.join(thin) or 'none'}")

Two things to read off that table before believing any single row.

The **n column comes first**. A class with a handful of validation articles can
post a perfect recall and mean nothing by it. The temporal split concentrates
the small classes down to almost nothing, so their scores are noisy by
construction - that is what the last line prints.

**Specificity is high everywhere**, and that is not a compliment. Each class is
a small slice of the corpus, so "not this topic" is an easy call. With 13
classes that is still more true than with six, and it is exactly the same reason
plain accuracy flatters this problem.

### 13 classes, one number: macro, micro or weighted

Three ways to collapse that table, answering three different questions.

- **Macro** - average the per-class scores, every class counting once. A class
  with 40 articles and a class with 400 get one vote each.
- **Micro** - pool all the TP, FP and FN first, *then* divide. Every article
  counts once, so the big classes decide the number. When every article gets
  exactly one label, micro-F1 is just accuracy.
- **Weighted** - per-class scores averaged, each weighted by how common its
  class is. Sits between the two.

I headline macro because the smallest class is the one most likely to be quietly
broken, and micro would bury it.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

for average in ("macro", "micro", "weighted"):
    p, r, f, _ = precision_recall_fscore_support(
        y_val, predicted, labels=labels, average=average, zero_division=0
    )
    print(f"{average:<10} precision {p:.3f}   recall {r:.3f}   F1 {f:.3f}")

micro_f1 = precision_recall_fscore_support(
    y_val, predicted, labels=labels, average="micro", zero_division=0
)[2]

# Not a coincidence, it is an identity for single-label problems.
assert abs(micro_f1 - results[best].accuracy) < 1e-9
print("\nmicro-F1 came out exactly equal to accuracy, as it always does here.\n")

print(f"{'majority':<20} accuracy {results['majority'].accuracy:.3f}"
      f"   macro-F1 {results['majority'].macro_f1:.3f}")
print(f"{best:<20} accuracy {results[best].accuracy:.3f}"
      f"   macro-F1 {results[best].macro_f1:.3f}")

## 7. Graphs

Four pictures: where the mistakes are, which classes each model is good at, how
well the model *ranks* rather than decides, and how much I gain by letting it
refuse to answer.

### 7a. Confusion matrix

Rows are the true topic, columns are what the model said, and each row is
normalised so it reads as "of the real sport articles, what share went where".
At 13 by 13 the cell numbers are readable, but the shading is still what
carries it, and the mistakes worth discussing are printed underneath.

Everything off the diagonal is a mistake, and *which* mistake matters. Two
classes whose boundary is genuinely hard to draw from a headline alone -
`politics` against `business_economy`, say, where a government's economic
policy is both - confusing each other says something about where the line
actually sits. Sport confused with education would say something is broken.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8.4, 7.6))
ConfusionMatrixDisplay.from_predictions(
    y_val, predicted, labels=labels, normalize="true",
    cmap="Greys", colorbar=False, include_values=False,
    xticks_rotation=90, ax=ax,
)
ax.set_title(f"{best} on validation", loc="left")
plt.tight_layout()
plt.show()

confused = Counter(
    (truth, guess) for truth, guess in zip(y_val, predicted, strict=True) if truth != guess
)
print("the mistakes it actually makes:\n")
for (truth, guess), n in confused.most_common(12):
    same_family = truth.split("_")[0] == guess.split("_")[0]
    print(f"  {truth:<24} called {guess:<24}{n:>4}x"
          f"{'   (same family)' if same_family else ''}")

### 7b. Which class is each model good at?

The single macro-F1 number hides this. Two models can score the same and be good
at completely different topics.

In [ ]:
order = [labels[i] for i in np.argsort(-f1)]
spots = np.arange(len(order))
height = 0.8 / len(results)

fig, ax = plt.subplots(figsize=(7.2, 8.0))
for i, (name, result) in enumerate(results.items()):
    ax.barh(spots + i * height, [result.per_class_f1[c] for c in order], height, label=name)
ax.set_yticks(spots + height * (len(results) - 1) / 2)
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel("F1")
ax.set_title("Per-class score, every rung", loc="left")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 7c. ROC and AUC

Everything so far describes the model at one setting: whichever class scores
highest wins. But these models don't emit a label, they emit a score per class,
and the label is just `argmax`. Move the cut and every number above moves too.

A ROC curve sweeps that cut across its whole range and plots, at each point, the
share of a class's articles it caught (**TPR**, which is recall) against the
share of everything else it wrongly grabbed (**FPR**, which is 1 - specificity).
One curve per class, one against the rest again. Both axes are computed inside a
single true class, which is why class imbalance can't inflate them the way it
inflates accuracy.

**AUC** is the area underneath, and it reads in plain English: pick one real
sport article and one non-sport article at random, and AUC is how often the
model gives the sport one the higher sport score. 0.5 is a coin toss, and a
curve below the diagonal means inverting the model would improve it.

Even 13 labelled curves clutter a legend, so all of them are drawn in grey
and only the best and worst three are named.

`majority` is not on this plot. It emits hard labels with nothing to sweep, so it
is a single point in ROC space, not a curve.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# LinearSVC has no predict_proba, so fall back to its margin. ROC only needs the ranking.
clf = fitted[best].named_steps["clf"]
scores = (fitted[best].predict_proba(x_val) if hasattr(clf, "predict_proba")
          else fitted[best].decision_function(x_val))
emitted = list(clf.classes_)
one_vs_rest = label_binarize(y_val, classes=emitted)

areas = {}
for i, topic in enumerate(emitted):
    if one_vs_rest[:, i].any():
        areas[topic] = roc_auc_score(one_vs_rest[:, i], scores[:, i])
named = sorted(areas, key=lambda t: -areas[t])
named = named[:3] + named[-3:]

fig, ax = plt.subplots(figsize=(5.6, 5.2))
for i, topic in enumerate(emitted):
    if topic not in areas:
        continue
    fpr, tpr, _ = roc_curve(one_vs_rest[:, i], scores[:, i])
    if topic in named:
        ax.plot(fpr, tpr, lw=1.6, label=f"{topic}  {areas[topic]:.3f}")
    else:
        ax.plot(fpr, tpr, lw=0.7, color="#cccccc", zorder=0)
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="coin toss  0.500")
ax.set_xlabel("false positive rate  (1 - specificity)")
ax.set_ylabel("true positive rate  (recall)")
ax.set_title(f"{best}, one curve per class", loc="left")
ax.legend(frameon=False, fontsize=7.5, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"macro AUC {np.mean(list(areas.values())):.3f}")
print("AUC judges the ordering only. A model can rank every article correctly and")
print("still report probabilities that are uniformly far too confident.")

### 7d. Where to put the cut, and when to refuse to answer

The threshold is a product decision, not a statistical one, and this project has
a specific one to make. A reader would much rather see an article marked
`unsorted` than filed confidently under the wrong topic. So the model is allowed
to decline: if its best score isn't high enough, it says nothing.

That trades **coverage**, how many articles get a topic at all, against
**accuracy on the ones it does file**. The curve below is that trade-off. The
right cut is wherever accuracy stops climbing fast enough to be worth the
articles being thrown away.

This one uses `hashing_sgd` rather than the best rung, on purpose. LinearSVC
emits a margin with no scale you can compare across classes, and ComplementNB's
probabilities pile up against 1.0, because naive Bayes is confidently wrong as
easily as it is confidently right. `modified_huber` was chosen for this rung
precisely so there would be a number worth thresholding.

In [ ]:
rung = "hashing_sgd"
classes = np.array(fitted[rung].named_steps["clf"].classes_)

probability = fitted[rung].predict_proba(x_val)
confidence = probability.max(axis=1)
guess = classes[probability.argmax(axis=1)]
correct = guess == np.array(y_val)

cuts = np.linspace(0.0, 0.95, 40)
coverage = np.array([(confidence >= c).mean() for c in cuts])
kept_accuracy = np.array([
    correct[confidence >= c].mean() if (confidence >= c).any() else np.nan for c in cuts
])

fig, ax = plt.subplots(figsize=(6.6, 3.2))
ax.plot(cuts, coverage, color="#999999", label="coverage: share still given a topic")
ax.plot(cuts, kept_accuracy, color="#000000", label="accuracy on the ones it files")
ax.set_xlabel("confidence required before the model commits")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"{rung}, validation\n")
for c in (0.0, 0.2, 0.4, 0.6, 0.8):
    kept = confidence >= c
    if not kept.any():
        continue
    print(f"cut {c:.1f}   files {kept.mean():>6.1%} of articles"
          f"   {correct[kept].mean():.3f} accurate on those"
          f"   {(~kept).sum():>4} sent to unsorted")

## 8. Conclusion

Two questions that decide whether the score above is real, and one about
which of these 13 classes still need work.


### 8a. Did it learn the topic, or the newspaper?

The strongest words per class. If sport is driven by *cricket*, *innings*,
*wicket*, good, it learned the subject. If it's driven by *pti*, *ist*,
*bengaluru* or a masthead, it learned to recognise the feed and the score is
worthless no matter how high it is.

In [ ]:
vec = fitted["tfidf_linear_svc"].named_steps["vec"]
svc = fitted["tfidf_linear_svc"].named_steps["clf"]
vocabulary = np.array(vec.get_feature_names_out())

for row, topic in zip(svc.coef_, svc.classes_, strict=True):
    print(f"{topic:<24} {', '.join(vocabulary[np.argsort(row)[-10:][::-1]])}")

### 8b. Does it work on a newspaper it has never seen?

Harder version of the same question. Train with one publisher removed entirely,
then test only on that publisher. If the score collapses, the model was leaning
on house style rather than subject matter.

It has to be a whole **publisher**, not one of its sections. A single section
feed contains a single class, and scoring 13 classes against one is arithmetic,
not evidence. I learned that the hard way: my first attempt held out
"Indian Express - Technology", scored 0.111, and looked like catastrophic
leakage. It was five classes with zero support dragging the macro average down.
Scoring is restricted to the classes the held-out publisher actually files.

In [ ]:
# Source names look like "The Indian Express — Technology", so the em dash splits off the publisher.
def publisher_of(source_name):
    return source_name.split(" — ")[0]


publisher = Counter(publisher_of(e.source_name) for e in data.train).most_common(1)[0][0]
held_out = [e for e in data.train if publisher_of(e.source_name) == publisher]
remaining = [e for e in data.train if publisher_of(e.source_name) != publisher]
present = sorted({e.topic for e in held_out})

unseen = build_model("tfidf_linear_svc")
unseen.fit([e.text for e in remaining], [e.topic for e in remaining])
guessed = unseen.predict([e.text for e in held_out])

unseen_f1 = f1_score([e.topic for e in held_out], guessed,
                     labels=present, average="macro", zero_division=0)
# Same class restriction on the normal run, so the two numbers are comparable.
seen_f1 = f1_score(y_val, predicted, labels=present, average="macro", zero_division=0)

print(f"held out : {publisher}  ({len(held_out)} articles,"
      f" {len(present)} classes)")
print(f"seen     : {seen_f1:.3f}")
print(f"unseen   : {unseen_f1:.3f}")
print(f"drop     : {seen_f1 - unseen_f1:.3f}")

### 8c. Which classes are still weak

The taxonomy is fixed now, so this is no longer "should these be split
further" - that question is closed (see the header of `taxonomy.yaml`).
What is still open is which of the 13 classes the model is worst at, and
whether that is a data problem or a definition problem.


In [ ]:
weakest = sorted(results[best].per_class_f1.items(), key=lambda kv: kv[1])[:6]
support = Counter(y_val)

print("the six weakest classes, and whether thinness explains them:\n")
print(f"  {'class':<24}{'F1':>7}{'val n':>8}{'train n':>9}")
for topic, score in weakest:
    print(f"  {topic:<24}{score:>7.2f}{support.get(topic, 0):>8}{dist.get(topic, 0):>9}")

print("\nA class with a healthy train count and a poor F1 is not short of data -")
print("it is short of a definition its siblings don't already cover.")

### What I would say if someone asked

**The model works, the taxonomy is fixed at 13 classes, and more labels have
stopped being the lever.**

- Every rung beats the majority floor by a mile, so there is real signal in a
  headline alone. That was not obvious at the start given how short the text is.
- **The labels were the project, and that phase is now over.** Weak labels from
  feed sections were free and capped the whole thing: they reach two thirds of
  the corpus and can never express crime, conflict or disaster, because no
  publisher runs those sections. Three rounds of hand labelling - 4,301 articles
  - built the class list. **Round 3 is the one that settled it**: 265 labels
  aimed squarely at the weakest classes moved macro-F1 from 0.678 to 0.671. A
  round that changes nothing is still a result. It says the bottleneck moved.
- **`society_lifestyle` is a definition problem, not a data problem.** It has
  261 labels and 185 training articles - the fifth largest class here - and it
  still scores 0.31. It is three unrelated things glued together: community and
  social issues, labour and work, and travel/food/lifestyle. The 26-class run
  scored those three separately at 0.29, 0.46 and 0.00. No amount of feeding
  fixes a class whose siblings already cover everything it means.
- **The model can now decline.** Each class has its own confidence cut, chosen
  on validation against an 80% precision target. It files 89.9% of articles and
  its accuracy on those goes from 0.758 to 0.809; the rest come out `unsorted`.
  Three classes reach that target at *no* cut, which is a worse finding than a
  low F1 - it means their confidence score carries no usable signal at all.
- **The 26-class experiment is what closed the taxonomy question, not what
  opened it.** Splitting cost nothing in resolution, but four of the finer
  classes (`business_companies`, `labour_work`, `lifestyle_living`,
  `society_community`) failed on both retrieval and classification at once -
  never linguistically distinct from their siblings - and every top confusion
  sat inside one of these 13 families anyway. So the taxonomy folded back to
  the level that was carrying the real signal, and it stays there: see the
  header of `taxonomy.yaml`.
- The leakage checks pass. Top features are subject words, not mastheads or
  wire agencies, and holding out The Indian Express entirely - 1,098 articles -
  costs 0.029 macro-F1.

What I would do next, in order:

1. Decide what `society_lifestyle` should be: split it, absorb it, narrow it to
   one meaning, or keep it and let it abstain. Not another labelling round -
   that has been tried, and it is the one option the evidence rules out.
2. Calibrate the near-duplicate grouping. The threshold was never the real
   problem: the banding is. 16 bands of 8 rows only proposes a pair when a whole
   band matches, so the pairs that are actually in doubt are never surfaced to
   be judged.
3. Open the test split, once, and report that number.

Until step 3, everything here is a working number, not a result.